<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/Lasyer_organisation_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt # for making figures
%matplotlib inline

In [2]:
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

--2026-01-22 17:37:55--  https://raw.githubusercontent.com/karpathy/makemore/master/names.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 228145 (223K) [text/plain]
Saving to: ‘names.txt’

names.txt           100%[===================>] 222.80K  --.-KB/s    in 0.004s  

2026-01-22 17:37:55 (54.6 MB/s) - ‘names.txt’ saved [228145/228145]



In [3]:

words = open("names.txt", "r").read().splitlines()

In [9]:
chars=sorted(set('.'.join(words)))
itos={i:j for i,j in enumerate(chars)}
stoi={j:i for i,j in enumerate(chars)}
vocab_size = len(itos)

In [29]:
def data_prep(data,block_size):
  data= ["."*block_size +word +"." for word in data]
  Y=[stoi[i] for ch in data for i in ch[block_size:]]
  X=[word[i:i+block_size] for word in data for i in range(len(word)-block_size)]
  X=[stoi[i] for x in X for i in x]
  X=torch.tensor(X).view(-1,block_size)
  Y=torch.tensor(Y)
  return X, Y
block_size=4
Xtr,Ytr=data_prep(words,block_size)

In [16]:
Xtr.shape

torch.Size([228146, 3])

In [22]:

n_embd = 3 # the dimensionality of the character embedding vectors
n_hidden = 10 # the number of neurons in the hidden layer of the MLP

In [58]:
class map:
  def __init__(self, orig, emb):
    self.matrix=torch.randn(orig,emb)
  def __call__(self, x):
    self.out=self.matrix[x]
    return self.out.view(-1, (len(self.matrix[1,:])*len(x[1,:])))




class linear:
  def __init__(self, fan_in,fan_out, bias = True):
    self.weights=torch.randn(fan_in,fan_out)/(fan_in**0.5) #kainin norm
    if bias:
      self.bias=torch.zeros(fan_out)
    else:
      self.bias=None
  def __call__(self, x):
    self.out=x@ self.weights + self.bias
    return self.out
  def parameters(self):
    return [self.weights]+ ([] if self.bias is None else [self.bias])


class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

In [57]:
map1=map(vocab_size,n_embd)
input= map1(Xtr)

layer1=linear(n_embd*block_size,n_hidden)
layer1(input).size()

torch.Size([228146, 10])

tensor([[ 0,  0,  0,  0],
        [ 0,  0,  0,  5],
        [ 0,  0,  5, 13],
        ...,
        [ 0, 26, 26, 25],
        [26, 26, 25, 26],
        [26, 25, 26, 24]])